# Close curvature-space pairs: 6--10 ply layups

This notebook lists pairs of rotation-distinct layups whose stacking sequence is rearranged but whose raw 3-D curvature vectors are close. A pair is compared only when it has the same inventory of ply angles, so the table isolates the effect of through-thickness ordering.

The separation is the Euclidean distance in the engineering-curvature vector $[K_{11}, K_{22}, K_{12}]$, in m$^{-1}$. Layups related only by a 90-degree in-plane rotation are collapsed before comparison.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

plotting_dir = Path.cwd()
if not (plotting_dir / 'generate_deformation_uniqueness.py').exists():
    plotting_dir = Path('Plotting')

sys.path.insert(0, str(plotting_dir.resolve()))
import generate_deformation_uniqueness as uniqueness

data_path = plotting_dir.parent / 'AbaqusSweep' / 'ABaqusData_with_computational_results.csv'
if not data_path.exists():
    data_path = Path('AbaqusSweep/ABaqusData_with_computational_results.csv')
data = pd.read_csv(data_path)


In [ ]:
# Edit these controls to broaden or narrow the review table.
MIN_PLIES = 6
MAX_PLIES = 10
CLOSE_DISTANCE_M_INV = 0.05

# Keep True to compare only layups with identical counts of 0, 90, +45, etc.
REQUIRE_MATCHING_PLY_INVENTORY = True

RESPONSE_COLUMNS = uniqueness.RESPONSE_COLUMNS
CURVATURE_COLUMNS = uniqueness.CURVATURE_COLUMNS

def ply_inventory(layup):
    """A rotation-canonical, order-independent ply-angle inventory."""
    return tuple(sorted(uniqueness.parse_layup(layup)))

def changed_positions(layup_a, layup_b):
    return sum(a != b for a, b in zip(uniqueness.parse_layup(layup_a),
                                         uniqueness.parse_layup(layup_b)))


In [ ]:
rows = []
for ply_count in range(MIN_PLIES, MAX_PLIES + 1):
    raw_group = data.loc[data['NumberOfPlies'] == ply_count]
    group, rotational_duplicates, _ = uniqueness.rotation_canonical_group(raw_group)
    group['Ply inventory'] = group['Layup'].map(ply_inventory)

    candidate_groups = group.groupby('Ply inventory', sort=False) if REQUIRE_MATCHING_PLY_INVENTORY else [(None, group)]
    for inventory, candidates in candidate_groups:
        response = candidates[RESPONSE_COLUMNS].to_numpy(dtype=float)
        curvature = candidates[CURVATURE_COLUMNS].to_numpy(dtype=float)
        for left, right in combinations(range(len(candidates)), 2):
            distance = float(np.linalg.norm(curvature[left] - curvature[right]))
            if distance > CLOSE_DISTANCE_M_INV:
                continue
            layup_a = candidates.iloc[left]['Layup']
            layup_b = candidates.iloc[right]['Layup']
            row = {
                'Ply count': ply_count,
                'Curvature-space distance (m^-1)': distance,
                'Changed stacking positions': changed_positions(layup_a, layup_b),
                'Ply inventory': inventory,
                'Layup A': layup_a,
                'Layup B': layup_b,
            }
            for index, name in enumerate(('e0_11', 'e0_22', 'e0_12', 'K_11', 'K_22', 'K_12')):
                scale = 1e6 if name.startswith('e0_') else 1.0
                unit = 'microstrain' if name.startswith('e0_') else 'm^-1'
                row[f'{name} A ({unit})'] = response[left, index] * scale
                row[f'{name} B ({unit})'] = response[right, index] * scale
            rows.append(row)

close_pairs = pd.DataFrame(rows).sort_values(
    ['Ply count', 'Curvature-space distance (m^-1)', 'Changed stacking positions']
).reset_index(drop=True)

output_path = plotting_dir / f'close_rearranged_curvature_pairs_{MIN_PLIES}_to_{MAX_PLIES}.csv'
close_pairs.to_csv(output_path, index=False)
print(f'Found {len(close_pairs)} close rearranged pairs at d <= {CLOSE_DISTANCE_M_INV:g} m^-1.')
print(f'Wrote: {output_path}')
display(close_pairs)


## FREEFREE temperature-gradient sweep for the closest 8-ply rearrangement

The pair below is the closest 8-ply, same-inventory rearrangement in raw curvature space. Edit `THERMAL_GRADIENTS_C_PER_M` to prescribe the gradients to solve. The FREEFREE field is controlled by midplane temperature and through-thickness gradient; the sink temperatures are retained only for solver-case metadata.

In [ ]:
# User controls
MIDPLANE_TEMPERATURE_C = 50.0
THERMAL_GRADIENTS_C_PER_M = [-25_000.0, -50_000.0, -75_000.0]
TRANSVERSE_RITZ_DEGREE = 5
TOP_SINK_TEMPERATURE_C = 0.0
BOTTOM_SINK_TEMPERATURE_C = 100.0

# This is the closest 8-ply pair identified in the curvature-space table.
CLOSE_8PLY_LAYUPS = {
    'A': [-45, 0, 45, 90, 90, 45, 0, -45],
    'B': [0, -45, 90, 45, 45, 90, -45, 0],
}


In [ ]:
# Import the project-local FREEFREE solver without executing its script entry point.
solver_dir = plotting_dir.parent / 'Computational Real'
if not solver_dir.exists():
    solver_dir = Path('Computational Real')
sys.path.insert(0, str(solver_dir.resolve()))
import FREEFREE as freefree

response_names = ('e0_11', 'e0_22', 'e0_12', 'K_11', 'K_22', 'K_12')
detailed_rows = []

for gradient in THERMAL_GRADIENTS_C_PER_M:
    for pair_member, layup in CLOSE_8PLY_LAYUPS.items():
        row = {
            'Pair member': pair_member,
            'Layup': uniqueness.format_layup(layup),
            'Midplane temperature (C)': MIDPLANE_TEMPERATURE_C,
            'Thermal gradient (C/m)': gradient,
            'Ritz degree': TRANSVERSE_RITZ_DEGREE,
        }
        try:
            result = freefree.solve_case(
                layup, TOP_SINK_TEMPERATURE_C, BOTTOM_SINK_TEMPERATURE_C,
                transverse_degree=TRANSVERSE_RITZ_DEGREE,
                midplane_temperature=MIDPLANE_TEMPERATURE_C,
                thermal_gradient=gradient,
            )
            response = result['response']
            row.update({
                'Converged': True,
                'Solver iterations': result['solver_iterations'],
                'Top surface temperature (C)': result['top_surface_temperature'],
                'Bottom surface temperature (C)': result['bottom_surface_temperature'],
                'Solver message': result['solution']['solver'].message,
            })
            for name, value in zip(response_names, response):
                unit = 'microstrain' if name.startswith('e0_') else 'm^-1'
                row[f'{name} ({unit})'] = value * 1e6 if name.startswith('e0_') else value
        except RuntimeError as error:
            row.update({'Converged': False, 'Solver iterations': np.nan, 'Solver message': str(error)})
        detailed_rows.append(row)

freefree_gradient_results = pd.DataFrame(detailed_rows)
detailed_output = plotting_dir / 'close_8ply_freefree_gradient_sweep.csv'
freefree_gradient_results.to_csv(detailed_output, index=False)
display(freefree_gradient_results)
print(f'Wrote detailed FREEFREE results: {detailed_output}')


In [ ]:
# Pairwise A--B response separation at each prescribed gradient.
summary_rows = []
for gradient, group in freefree_gradient_results.groupby('Thermal gradient (C/m)', sort=True):
    successful = group.loc[group['Converged']].set_index('Pair member')
    row = {'Thermal gradient (C/m)': gradient}
    if {'A', 'B'}.issubset(successful.index):
        strain_columns = [f'{name} (microstrain)' for name in response_names[:3]]
        curvature_columns = [f'{name} (m^-1)' for name in response_names[3:]]
        strain_difference = successful.loc['A', strain_columns].to_numpy(float) - successful.loc['B', strain_columns].to_numpy(float)
        curvature_difference = successful.loc['A', curvature_columns].to_numpy(float) - successful.loc['B', curvature_columns].to_numpy(float)
        row['Strain-vector separation (microstrain)'] = np.linalg.norm(strain_difference)
        row['Curvature-vector separation (m^-1)'] = np.linalg.norm(curvature_difference)
        row['Both cases converged'] = True
    else:
        row.update({'Strain-vector separation (microstrain)': np.nan,
                    'Curvature-vector separation (m^-1)': np.nan,
                    'Both cases converged': False})
    summary_rows.append(row)

freefree_gradient_separation_summary = pd.DataFrame(summary_rows)
summary_output = plotting_dir / 'close_8ply_freefree_gradient_separation_summary.csv'
freefree_gradient_separation_summary.to_csv(summary_output, index=False)
display(freefree_gradient_separation_summary)
print(f'Wrote A--B separation summary: {summary_output}')
